In [40]:
import math
import random
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from pathlib import Path

SEED = 7
DATA_DIR = Path('shakespeare-dataset-main/text')
BLOCK_SIZE, BATCH_SIZE, NUM_WORKERS = 64, 128, 0
DIM, NUM_LAYERS, NUM_HEADS, FF_DIM, DROPOUT = 16, 12, 8, 1344, 0.1
DIFFUSION_STEPS, SCHEDULE_OFFSET = 2000, 1e-4
LEARNING_RATE, WEIGHT_DECAY, GRAD_CLIP_NORM = 3e-4, 1e-2, 1.0
TRAIN_STEPS, LOG_EVERY, SAMPLE_EVERY = 10_000, 50, 1_000
MSE_WEIGHT, ROUNDING_WEIGHT = 1.0, 1.0
TRACE_POINTS = 8
CLAMP_START_T = DIFFUSION_STEPS  # Paper-style all-step clamping; compare_clamping() tests alternatives.
SAMPLE_BATCH_SIZE, SAMPLE_REVERSE_STEPS = 2, 250

device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)
random.seed(SEED)
print(f'Using {device}')

Using mps


In [41]:
def clean_folger_text(raw):
    lines = raw.replace('\r\n', '\n').replace('\r', '\n').splitlines(keepends=True)
    return ''.join(lines[7:]).lstrip('\n')

paths = sorted(DATA_DIR.glob('*.txt'))
if len(paths) < 2:
    raise FileNotFoundError(f'Need at least two .txt files in {DATA_DIR}')
train_paths, val_paths = paths[:-4], paths[-4:]
train_text = '\n\n'.join(clean_folger_text(p.read_text(encoding='utf-8')) for p in train_paths)
val_text = '\n\n'.join(clean_folger_text(p.read_text(encoding='utf-8')) for p in val_paths)
chars = sorted(set(train_text + val_text))
stoi = {char: index for index, char in enumerate(chars)}
itos = dict(enumerate(chars))
train_tokens = torch.tensor([stoi[c] for c in train_text], dtype=torch.long)
val_tokens = torch.tensor([stoi[c] for c in val_text], dtype=torch.long)

class CharacterBlocks(Dataset):
    def __init__(self, values, block_size):
        self.values = values
        self.block_size = block_size
    def __len__(self): return len(self.values) // self.block_size
    def __getitem__(self, index):
        start = index * self.block_size
        return self.values[start:start + self.block_size]

train_dataset = CharacterBlocks(train_tokens, BLOCK_SIZE)
val_dataset = CharacterBlocks(val_tokens, BLOCK_SIZE)
loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=NUM_WORKERS)
print(f'Train: {len(train_paths)} works, {len(train_text):,} chars | validation: {len(val_paths)} works, {len(val_text):,} chars | vocab={len(chars)}')

Train: 38 works, 4,868,481 chars | validation: 4 works, 450,656 chars | vocab=84


In [42]:
class DiffusionLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, DIM)
        self.position = nn.Embedding(BLOCK_SIZE, DIM)
        frequencies = torch.exp(-math.log(10_000) * torch.arange(DIM // 2) / (DIM // 2))
        self.register_buffer('time_frequencies', frequencies)
        layer = nn.TransformerEncoderLayer(DIM, NUM_HEADS, FF_DIM, DROPOUT, batch_first=True, activation='gelu')
        self.transformer = nn.TransformerEncoder(layer, NUM_LAYERS)
        self.final_norm = nn.LayerNorm(DIM)
        self.x0_head = nn.Linear(DIM, DIM)
        self.rounding_bias = nn.Parameter(torch.zeros(vocab_size))

    def forward(self, x_t, timesteps):
        pos = torch.arange(x_t.size(1), device=x_t.device)
        angles = timesteps[:, None] * self.time_frequencies
        time_embedding = torch.cat((angles.sin(), angles.cos()), dim=-1)
        hidden = x_t + self.position(pos).unsqueeze(0) + time_embedding.unsqueeze(1)
        return self.x0_head(self.final_norm(self.transformer(hidden)))

    def nearest_tokens(self, vectors):
        distances = torch.cdist(vectors.reshape(-1, vectors.size(-1)), self.embedding.weight)
        return distances.argmin(-1).view(vectors.shape[:-1])

    def nearest_embedding(self, vectors):
        return self.embedding(self.nearest_tokens(vectors))

    def rounding_logits(self, vectors):
        return F.linear(vectors, self.embedding.weight, self.rounding_bias)

    def round_to_tokens(self, vectors):
        return self.rounding_logits(vectors).argmax(-1)

model = DiffusionLM(len(chars)).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

Parameters: 548,996


In [43]:
# Match the released sqrt beta schedule, including its clipped final beta.
# Build in float64 on CPU for precision; MPS only supports float32 tensors.
progress = torch.arange(DIFFUSION_STEPS + 1, dtype=torch.float64) / DIFFUSION_STEPS
schedule_curve = 1 - torch.sqrt(progress + SCHEDULE_OFFSET)
betas = (1 - schedule_curve[1:] / schedule_curve[:-1]).clamp(max=0.999)
alpha_bar = torch.cat((torch.ones(1, dtype=torch.float64), torch.cumprod(1 - betas, 0))).to(device=device, dtype=torch.float32)
# The paper's q_phi(x0 | w) uses the first diffusion step's noise scale.
EMBEDDING_STD = (1 - alpha_bar[1]).sqrt().item()
print(f'q_phi embedding noise std={EMBEDDING_STD:.4f} | terminal alpha_bar={alpha_bar[-1].item():.2e}')

def sample_x0(ids):
    # Reparameterized draw from q_phi(x0 | w); this is in addition to q(x_t | x0).
    return model.embedding(ids) + EMBEDDING_STD * torch.randn(*ids.shape, DIM, device=ids.device)

def q_sample(x0, t):
    alpha = alpha_bar[t].view(-1, 1, 1)
    return alpha.sqrt() * x0 + (1 - alpha).sqrt() * torch.randn_like(x0)

def loss_terms(ids):
    x0 = sample_x0(ids)
    t = torch.randint(1, DIFFUSION_STEPS + 1, (ids.size(0),), device=device)
    prediction = model(q_sample(x0, t), t)
    target = torch.where((t == 1).view(-1, 1, 1), model.embedding(ids), x0)
    mse = F.mse_loss(prediction, target)
    rounding = F.cross_entropy(model.rounding_logits(x0).reshape(-1, len(chars)), ids.reshape(-1))
    prior = alpha_bar[-1] * x0.square().mean()
    weighted_mse = MSE_WEIGHT * mse
    weighted_rounding = ROUNDING_WEIGHT * rounding
    return weighted_mse + weighted_rounding + prior, weighted_mse, weighted_rounding, prior

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

q_phi embedding noise std=0.1210 | terminal alpha_bar=2.02e-07


In [44]:
def step(batch):
    optimizer.zero_grad(set_to_none=True)
    values = loss_terms(batch.to(device))
    values[0].backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
    optimizer.step()

    return *values, grad_norm

def train(steps=TRAIN_STEPS):
    model.train()
    iterator = iter(loader)
    running = torch.zeros(5)
    for number in range(1, steps + 1):
        try: batch = next(iterator)
        except StopIteration:
            iterator = iter(loader)
            batch = next(iterator)
        values = step(batch)
        running += torch.tensor([value.item() for value in values])
        if number % LOG_EVERY == 0 or number == steps:
            mean = running / min(LOG_EVERY, number)
            print(
                f'step {number}/{steps} | total={mean[0]:.4f} | '
                f'weighted_mse={mean[1]:.4f} | weighted_rounding={mean[2]:.4f} | '
                f'prior={mean[3]:.2e} | grad_norm={mean[4]:.4f}'
            )
            running.zero_()
        if number % SAMPLE_EVERY == 0:
            print(f'\nSamples at step {number}:')
            blocks, norms, trace = sample(return_norms=True, return_trace=True)
            print('Reverse norm²/d | ' + ' | '.join(f't={t}: {value:.3f}' for t, value in norms))
            for row in trace:
                print(f"t={row['t']:4d}->{row['next_t']:4d} | pred_norm²/d={row['pred_norm']:.3f} | "
                      f"nearest_delta²/d={row['clamp_delta']:.3f} | clamp={'on' if row['clamp'] else 'off'} | changed={row['changed']}")
                print(f"  predicted: {row['predicted']!r}")
                print(f"  nearest:   {row['nearest']!r}")
            print('Final samples:')
            for token_block in blocks.cpu():
                print(''.join(itos[token.item()] for token in token_block))
                print()

In [45]:
@torch.no_grad()
def sample(batch_size=SAMPLE_BATCH_SIZE, reverse_steps=SAMPLE_REVERSE_STEPS,
           return_norms=False, return_trace=False, trace_points=TRACE_POINTS,
           clamp_start_t=CLAMP_START_T, seed=None):
    if not 1 <= reverse_steps <= DIFFUSION_STEPS:
        raise ValueError('reverse_steps must be between 1 and DIFFUSION_STEPS')
    if clamp_start_t is not None and not 0 <= clamp_start_t <= DIFFUSION_STEPS:
        raise ValueError('clamp_start_t must be None or between 0 and DIFFUSION_STEPS')
    if return_trace and trace_points < 1:
        raise ValueError('trace_points must be positive')
    # A CPU generator gives paired comparisons the same noise on CPU, CUDA, and MPS.
    cpu_rng = torch.Generator().manual_seed(seed) if seed is not None else None
    def draw_noise(shape):
        if cpu_rng is None:
            return torch.randn(shape, device=device)
        return torch.randn(shape, generator=cpu_rng).to(device)

    was_training = model.training
    model.eval()
    try:
        x_t = draw_noise((batch_size, BLOCK_SIZE, DIM))
        norms = [(DIFFUSION_STEPS, x_t.square().mean().item())] if return_norms else None
        trace = [] if return_trace else None
        previous_ids = None
        times = torch.linspace(DIFFUSION_STEPS, 0, reverse_steps + 1, device=device).long()
        trace_stride = max(1, reverse_steps // trace_points)
        for step, (current, next_time) in enumerate(zip(times[:-1], times[1:]), 1):
            t = torch.full((batch_size,), int(current), device=device, dtype=torch.long)
            predicted_x0 = model(x_t, t)
            apply_clamp = clamp_start_t is not None and int(current) <= clamp_start_t
            if apply_clamp or return_trace:
                nearest_ids = model.nearest_tokens(predicted_x0)
                nearest_x0 = model.embedding(nearest_ids)
            used_x0 = nearest_x0 if apply_clamp else predicted_x0
            if return_trace and (step == 1 or step % trace_stride == 0 or step == reverse_steps):
                raw_ids = model.round_to_tokens(predicted_x0[0])
                used_ids = nearest_ids[0] if apply_clamp else raw_ids
                changed = '—' if previous_ids is None else f'{(used_ids != previous_ids).float().mean().item():.1%}'
                trace.append({
                    't': int(current), 'next_t': int(next_time), 'clamp': apply_clamp,
                    'pred_norm': predicted_x0[0].square().mean().item(),
                    'clamp_delta': (nearest_x0[0] - predicted_x0[0]).square().mean().item(),
                    'changed': changed,
                    'predicted': ''.join(itos[i] for i in raw_ids.cpu().tolist()),
                    'nearest': ''.join(itos[i] for i in nearest_ids[0].cpu().tolist()),
                })
                previous_ids = used_ids.clone()
            a_t, a_next = alpha_bar[current], alpha_bar[next_time]
            step_alpha = a_t / a_next
            denominator = (1 - a_t).clamp_min(1e-8)
            mean = (a_next.sqrt() * (1 - step_alpha) / denominator) * used_x0
            mean += (step_alpha.sqrt() * (1 - a_next) / denominator) * x_t
            variance = ((1 - a_next) * (1 - step_alpha) / denominator).clamp_min(0)
            x_t = mean + variance.sqrt() * draw_noise(x_t.shape)
            if return_norms and step in (reverse_steps // 2, reverse_steps * 9 // 10, reverse_steps):
                norms.append((int(next_time), x_t.square().mean().item()))
        result = model.round_to_tokens(x_t)
        if return_norms and return_trace:
            return result, norms, trace
        if return_norms:
            return result, norms
        if return_trace:
            return result, trace
        return result
    finally:
        model.train(was_training)

In [46]:
train()


step 50/10000 | total=1.1100 | weighted_mse=1.0497 | weighted_rounding=0.0604 | prior=2.28e-07 | grad_norm=0.4326
step 100/10000 | total=0.9512 | weighted_mse=0.9023 | weighted_rounding=0.0489 | prior=2.25e-07 | grad_norm=0.2412
step 150/10000 | total=0.8730 | weighted_mse=0.8324 | weighted_rounding=0.0406 | prior=2.21e-07 | grad_norm=0.2626


KeyboardInterrupt: 

In [ ]:
# Rerun this cell to inspect the current model without retraining.
blocks, norms, trace = sample(return_norms=True, return_trace=True)
print('Reverse norm²/d | ' + ' | '.join(f't={t}: {value:.3f}' for t, value in norms))
for row in trace:
    print(f"t={row['t']:4d}->{row['next_t']:4d} | pred_norm²/d={row['pred_norm']:.3f} | "
          f"nearest_delta²/d={row['clamp_delta']:.3f} | clamp={'on' if row['clamp'] else 'off'} | changed={row['changed']}")
    print(f"  predicted: {row['predicted']!r}")
    print(f"  nearest:   {row['nearest']!r}")
print('Final samples:')
for token_block in blocks.cpu():
    print(''.join(itos[token.item()] for token in token_block))
    print()

In [ ]:
# Optional: compare clamping choices with identical initial and reverse-process noise.
def compare_clamping(reverse_steps=100, seed=1234):
    for label, start_t in [('all steps', DIFFUSION_STEPS), ('late steps', DIFFUSION_STEPS // 4), ('disabled', None)]:
        block = sample(batch_size=1, reverse_steps=reverse_steps, clamp_start_t=start_t, seed=seed)[0]
        text = ''.join(itos[i] for i in block.cpu().tolist())
        print(f'{label} (clamp_start_t={start_t}): {text!r}')

# Run compare_clamping() after training or after loading a checkpoint.


In [ ]:
# Fixed blocks make results more comparable across checkpoints.
eval_generator = torch.Generator().manual_seed(1234)
eval_indices = torch.randperm(
    len(val_dataset), generator=eval_generator
)[:min(64, len(val_dataset))].tolist()
eval_blocks = torch.stack([val_dataset[i] for i in eval_indices])

@torch.no_grad()
def reconstruction_diagnostics(
    blocks=eval_blocks,
    timesteps=(1, 10, 100, 250, 500, 1000, 1500, 2000),
):
    was_training = model.training
    model.eval()
    try:
        ids = blocks.to(device)
        x0 = sample_x0(ids)
        results = []

        for timestep in timesteps:
            t = torch.full(
                (ids.size(0),), timestep, device=device, dtype=torch.long
            )
            prediction = model(q_sample(x0, t), t)
            mse = F.mse_loss(prediction, x0).item()
            accuracy = (
                model.round_to_tokens(prediction) == ids
            ).float().mean().item()
            results.append((timestep, mse, accuracy))

        return results
    finally:
        model.train(was_training)

for t, mse, accuracy in reconstruction_diagnostics():
    print(f"t={t:4d} | mse={mse:.4f} | token_accuracy={accuracy:.1%}")

In [ ]:
# Compare generated character frequencies with held-out Shakespeare text.
@torch.no_grad()
def sample_diagnostics(num_samples=16):
    generated = sample(batch_size=num_samples, reverse_steps=SAMPLE_REVERSE_STEPS)
    generated_counts = torch.bincount(generated.flatten().cpu(), minlength=len(chars)).float()
    reference_counts = torch.bincount(val_tokens, minlength=len(chars)).float()
    generated_probs = generated_counts / generated_counts.sum()
    reference_probs = reference_counts / reference_counts.sum()
    print(f'Generated vs held-out character-frequency L1: {(generated_probs - reference_probs).abs().sum().item():.3f}')
    print(f'Generated unique characters: {(generated_counts > 0).sum().item()}/{len(chars)}')

sample_diagnostics()

In [ ]:
checkpoint_path = Path('checkpoints/dlm_5m.pt')
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    'model': model.state_dict(),
    'optimizer': optimizer.state_dict(),
    'chars': chars,
    'config': {
        'dim': DIM, 'block_size': BLOCK_SIZE, 'diffusion_steps': DIFFUSION_STEPS,
        'schedule_offset': SCHEDULE_OFFSET, 'embedding_std': EMBEDDING_STD,
        'train_steps': TRAIN_STEPS, 'seed': SEED,
    },
}, checkpoint_path)
print(f'Saved {checkpoint_path.resolve()}')